# NYU
Trained only on the provided training images, no pretrained weights.

In [1]:
import os, time, csv, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev=='cpu': torch.set_num_threads(os.cpu_count())
print('device:', dev)
# locate the competition data automatically
hits = glob.glob('/kaggle/input/**/train_labels.csv', recursive=True) or glob.glob('./**/train_labels.csv', recursive=True)
D = os.path.dirname(hits[0]) + '/'
print('data dir:', D)

device: cuda
data dir: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl/


In [2]:
tr = list(csv.DictReader(open(D+'train_labels.csv')))
te = list(csv.DictReader(open(D+'test.csv')))
load = lambda folder, rows: np.stack([np.array(Image.open(f'{D}{folder}/{r["filename"]}').convert('RGB')) for r in rows])
X = load('train_images', tr); y = np.array([int(r['label']) for r in tr]); Xt = load('test_images', te)
print(X.shape, Xt.shape, np.bincount(y))

(50000, 32, 32, 3) (2000, 32, 32, 3) [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]


In [3]:
mean = torch.tensor(X.reshape(-1,3).mean(0)).view(1,3,1,1).float(); std = torch.tensor(X.reshape(-1,3).std(0)).view(1,3,1,1).float()
prep = lambda a: ((torch.from_numpy(a).permute(0,3,1,2).float()-mean)/std).contiguous(memory_format=torch.channels_last)
perm = np.random.permutation(len(X)); vi, ti = perm[:2000], perm[2000:]   # 2000-image holdout to measure accuracy
Xtr, ytr = prep(X[ti]).to(dev), torch.from_numpy(y[ti]).to(dev)
Xva, yva = prep(X[vi]).to(dev), torch.from_numpy(y[vi]).to(dev)
Xte = prep(Xt).to(dev)

In [4]:
def cb(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
class Res(nn.Module):
    def __init__(s,c): super().__init__(); s.a=cb(c,c); s.b=cb(c,c)
    def forward(s,x): return x + s.b(s.a(x))          # skip connection
w = (32,64,128,256)
m = nn.Sequential(cb(3,w[0]), cb(w[0],w[1]), nn.MaxPool2d(2), Res(w[1]), cb(w[1],w[2]), nn.MaxPool2d(2),
                  cb(w[2],w[3]), nn.MaxPool2d(2), Res(w[3]), nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(w[3],10)
                 ).to(dev).to(memory_format=torch.channels_last)
print('parameters:', sum(p.numel() for p in m.parameters()))

parameters: 1646122


In [5]:
EP, BS = 34, 256
n = len(Xtr); spe = n // BS
opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=EP*spe, pct_start=0.25, anneal_strategy='linear')

def aug(xb):  # random crop (pad 4) + horizontal flip + 8x8 cutout
    B = xb.size(0); p = F.pad(xb,(4,4,4,4),mode='reflect'); out = torch.empty_like(xb)
    dx = np.random.randint(0,9,B); dy = np.random.randint(0,9,B); fl = np.random.rand(B)<.5
    cx = np.random.randint(0,32,B); cy = np.random.randint(0,32,B)
    for i in range(B):
        t = p[i,:,dy[i]:dy[i]+32,dx[i]:dx[i]+32]
        out[i] = t.flip(2) if fl[i] else t
        out[i,:,max(0,cy[i]-4):cy[i]+4,max(0,cx[i]-4):cx[i]+4] = 0
    return out.contiguous(memory_format=torch.channels_last)

def probs(Xs, tta=True):
    m.eval(); o=[]
    with torch.no_grad():
        for i in range(0,len(Xs),500):
            b = Xs[i:i+500]; l = m(b).softmax(1)
            if tta: l = l + m(b.flip(3)).softmax(1)
            o.append(l)
    return torch.cat(o)

t0 = time.time()
for ep in range(EP):
    m.train(); idx = torch.randperm(n, device=dev); tl = 0
    for s in range(spe):
        b = idx[s*BS:(s+1)*BS]; xb = aug(Xtr[b]); yb = ytr[b]
        loss = F.cross_entropy(m(xb), yb, label_smoothing=0.1)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step(); tl += loss.item()
    va = (probs(Xva,False).argmax(1)==yva).float().mean().item()
    print(f'ep {ep+1}/{EP} loss {tl/spe:.3f} val_acc {va:.4f} elapsed {(time.time()-t0)/60:.1f}m')

ep 1/34 loss 1.725 val_acc 0.6355 elapsed 0.3m
ep 2/34 loss 1.383 val_acc 0.6930 elapsed 0.6m
ep 3/34 loss 1.219 val_acc 0.7300 elapsed 0.9m
ep 4/34 loss 1.130 val_acc 0.7060 elapsed 1.2m
ep 5/34 loss 1.051 val_acc 0.7345 elapsed 1.5m
ep 6/34 loss 0.993 val_acc 0.7860 elapsed 1.8m
ep 7/34 loss 0.955 val_acc 0.7975 elapsed 2.0m
ep 8/34 loss 0.923 val_acc 0.8310 elapsed 2.3m
ep 9/34 loss 0.896 val_acc 0.7785 elapsed 2.6m
ep 10/34 loss 0.870 val_acc 0.8320 elapsed 2.9m
ep 11/34 loss 0.842 val_acc 0.8480 elapsed 3.2m
ep 12/34 loss 0.826 val_acc 0.8305 elapsed 3.5m
ep 13/34 loss 0.811 val_acc 0.8525 elapsed 3.8m
ep 14/34 loss 0.795 val_acc 0.8630 elapsed 4.1m
ep 15/34 loss 0.781 val_acc 0.8510 elapsed 4.4m
ep 16/34 loss 0.767 val_acc 0.8425 elapsed 4.6m
ep 17/34 loss 0.758 val_acc 0.8600 elapsed 4.9m
ep 18/34 loss 0.746 val_acc 0.8845 elapsed 5.2m
ep 19/34 loss 0.737 val_acc 0.8830 elapsed 5.5m
ep 20/34 loss 0.731 val_acc 0.8860 elapsed 5.8m
ep 21/34 loss 0.719 val_acc 0.8555 elapsed 6.1m
e

In [6]:
print('Holdout accuracy (with flip TTA):', (probs(Xva,True).argmax(1)==yva).float().mean().item())
pred = probs(Xte,True).argmax(1).cpu().numpy()
with open('submission.csv','w',newline='') as f:
    wr = csv.writer(f); wr.writerow(['id','label'])
    for r,p in zip(te,pred): wr.writerow([r['id'], int(p)])
print('wrote submission.csv', len(pred), 'rows', np.bincount(pred))

Holdout accuracy (with flip TTA): 0.9355000257492065
wrote submission.csv 2000 rows [168 219 200 216 196 218 177 198 208 200]
